# Лабораторная работа – Эмбеддинги слов

## Библиотека Gensim для построения модели Word2Vec

In [ ]:
import re  
import pandas as pd  
from time import time 
from gensim.models import Word2Vec

In [ ]:
# Загрузка данных для обучения
df = pd.read_csv('dataset.csv')
df.shape

In [ ]:
# построение словаря для обучения
t = time()
w2v_model.build_vocab(sentences, progress_per=10000)
print('Время, потраченное на построение словаря, составило: {} минут'.format(round((time() - t) / 60, 2)))

In [ ]:
# построение модели
w2v_model = Word2Vec(min_count=20,      # минимальная частота слова (2-100)
                     window=2,          # длина окна (2-10)
                     size=300,          # размер эмбеддинга (50-300)
                     sample=6e-5,       # 
                     alpha=0.03,        # learning rate
                     min_alpha=0.0007,  # 
                     negative=20)       # величина отрицательной выборки (0, 5-20)

In [ ]:
# Обучение модели
t = time()
w2v_model.train(sentences, total_examples=w2v_model.corpus_count, epochs=30, report_delay=1)
print('Время, потраченное на обучение модели, составило: {} минут'.format(round((time() - t) / 60, 2)))

In [ ]:
# Если не обучать модель дальше, то можно оптимизировать использование памяти
w2v_model.init_sims(replace=True)

In [ ]:
# Использование модели:
# самые близкие слова
w2v_model.wv.most_similar(positive=["человек"]) 
# арифметические выражения над векторами
w2v_model.wv.most_similar(positive=["король", "женщина"], negative=["мужчина"], topn=3)
# мера близости слов
w2v_model.wv.similarity("кафе", 'ресторан') 
# выбрать лишнее слово
w2v_model.wv.doesnt_match(["кафе", "ресторан", "велосипед"]) 

## Визуализация векторов в двумерном пространстве

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
 
import seaborn as sns
sns.set_style("darkgrid")

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [ ]:
def tsnescatterplot(model, word, list_names):
    """ Plot in seaborn the results from the t-SNE dimensionality reduction algorithm of the vectors of a query word,
    its list of most similar words, and a list of words.
    """
    arrays = np.empty((0, 300), dtype='f')
    word_labels = [word]
    color_list  = ['red']

    # adds the vector of the query word
    arrays = np.append(arrays, model.wv.__getitem__([word]), axis=0)
    
    # gets list of most similar words
    close_words = model.wv.most_similar([word])
    
    # adds the vector for each of the closest words to the array
    for wrd_score in close_words:
        wrd_vector = model.wv.__getitem__([wrd_score[0]])
        word_labels.append(wrd_score[0])
        color_list.append('blue')
        arrays = np.append(arrays, wrd_vector, axis=0)
    
    # adds the vector for each of the words from list_names to the array
    for wrd in list_names:
        wrd_vector = model.wv.__getitem__([wrd])
        word_labels.append(wrd)
        color_list.append('green')
        arrays = np.append(arrays, wrd_vector, axis=0)
        
    # Reduces the dimensionality from 300 to 50 dimensions with PCA
    reduc = PCA(n_components=50).fit_transform(arrays)
    
    # Finds t-SNE coordinates for 2 dimensions
    np.set_printoptions(suppress=True)
    
    Y = TSNE(n_components=2, random_state=0, perplexity=15).fit_transform(reduc)
    
    # Sets everything up to plot
    df = pd.DataFrame({'x': [x for x in Y[:, 0]],
                       'y': [y for y in Y[:, 1]],
                       'words': word_labels,
                       'color': color_list})
    
    fig, _ = plt.subplots()
    fig.set_size_inches(9, 9)
    
    # Basic plot
    p1 = sns.regplot(data=df,
                     x="x",
                     y="y",
                     fit_reg=False,
                     marker="o",
                     scatter_kws={'s': 40,
                                  'facecolors': df['color']
                                 }
                    )
    
    # Adds annotations one by one with a loop
    for line in range(0, df.shape[0]):
         p1.text(df["x"][line],
                 df['y'][line],
                 '  ' + df["words"][line].title(),
                 horizontalalignment='left',
                 verticalalignment='bottom', size='medium',
                 color=df['color'][line],
                 weight='normal'
                ).set_size(15)

    
    plt.xlim(Y[:, 0].min()-50, Y[:, 0].max()+50)
    plt.ylim(Y[:, 1].min()-50, Y[:, 1].max()+50)
            
    plt.title('t-SNE визуализация для {}'.format(word.title()))

In [ ]:
tsnescatterplot(w2v_model, 'животное', ['собака', 'кошка', 'жираф', 'змея', 'бабочка', 'жук', 'лодка', 'банан'])
tsnescatterplot(w2v_model, 'дерево', [i[0] for i in w2v_model.wv.most_similar(negative=["дерево"])])
tsnescatterplot(w2v_model, "посуда", [t[0] for t in w2v_model.wv.most_similar(positive=["посуда"], topn=20)][10:])

## Библиотека Gensim для построения модели FastText

In [ ]:
from gensim.models.fasttext import FastText as FT_gensim
model_gensim = FT_gensim(size=100)

# построение словаря для обучения
model_gensim.build_vocab(corpus_file=corpus_file)

# train the model
# - model: Training architecture. Allowed values: `cbow`, `skipgram` (Default `cbow`)
# - size: Size of embeddings to be learnt (Default 100)
# - alpha: Initial learning rate (Default 0.025)
# - window: Context window size (Default 5)
# - min_count: Ignore words with number of occurrences below this (Default 5)
# - loss: Training objective. Allowed values: `ns`, `hs`, `softmax` (Default `ns`)
# - sample: Threshold for downsampling higher-frequency words (Default 0.001)
# - negative: Number of negative words to sample, for `ns` (Default 5)
# - iter: Number of epochs (Default 5)
# - sorted_vocab: Sort vocab by descending frequency (Default 1)
# - threads: Number of threads to use (Default 12)
# - min_n: min length of char ngrams (Default 3)
# - max_n: max length of char ngrams (Default 6)
# - bucket: number of buckets used for hashing ngrams (Default 2000000)
    
model_gensim.train(
    corpus_file=corpus_file, epochs=model_gensim.epochs,
    total_examples=model_gensim.corpus_count, total_words=model_gensim.corpus_total_words
)

print(model_gensim)
# сохранение обученной модели
model_gensim.save('saved_model_gensim')
loaded_model = FT_gensim.load('saved_model_gensim')
print(loaded_model)

# сохранение обученной модели с использованием fastText wrapper
model_wrapper.save('saved_model_wrapper')
loaded_model = FT_wrapper.load('saved_model_wrapper')
print(loaded_model)

# Проверка наличия слова в словаре
print('человек' in model_wrapper.wv.vocab)
print('люди' in model_wrapper.wv.vocab)
print(model_wrapper['человек'])
print(model_wrapper['люди'])
print("колобок" in model_wrapper)
# Использование модели:
# самые близкие слова
model_wrapper.most_similar(positive=["человек"]) 
# арифметические выражения над векторами
model_wrapper.most_similar(positive=["король", "женщина"], negative=["мужчина"], topn=3)
# мера близости слов
model_wrapper.similarity("кафе", 'ресторан') 
# выбрать лишнее слово
model_wrapper.doesnt_match(["кафе", "ресторан", "велосипед"]) 
# мера близости слов
model_wrapper.wmdistance(sentence_1, sentence_2)


## Предобученные модели Facebook

In [ ]:
!wget https://github.com/facebookresearch/fastText/archive/v0.9.2.zip
!unzip v0.9.2.zip
!cd fastText-0.9.2
!pip install fasttext


import fasttext
import fasttext.util
fasttext.util.download_model('ru', if_exists='ignore')  # 
ft = fasttext.load_model('cc.ru.300.bin')
fasttext.util.reduce_model(ft, 100) # адаптировать размер модели
# или обучить: 
# Сначала текст очищается от всех символов, кроме букв, 
# приводится в нижний регистр, 
# делается стемминг или лемматизация. 
# В модель текст подается токенизированным (разбитым на слова).
ft = fasttext.train_unsupervised('data/fil9') # skipgram
ft = fasttext.train_unsupervised('data/fil9', "cbow") # cbow
# использование модели
# размерность вектора эмбеддинга
ft.get_dimension()
# получить вектор
ft.get_word_vector('веселый').shape
# ближайшие слова
ft.get_nearest_neighbors('веселый')
# найти слова для слова "франция", такие же, каким является слово "берлин" для слова "германия"
ft.get_analogies("берлин", "германия", "франция")
# сохранить модель
ft.save_model('cc.ru.100.bin')


## Предобученные модели ELMo от AllenNLP

In [ ]:
!pip install allennlp-models
from allennlp_models import pretrained
pretrained.load_predictor("mc-roberta-swag")

## Предобученные модели от RusVectors

In [ ]:
import wget
import zipfile
import gensim

model_url = 'http://vectors.nlpl.eu/repository/20/183.zip'
wget.download(model_url)
model_file = model_url.split('/')[-1]
with zipfile.ZipFile(model_file, 'r') as archive:
    stream = archive.open('model.bin')
    model_vec = gensim.models.KeyedVectors.load_word2vec_format(stream, binary=True)
print('vector model is redy!')

print(model_vec.get_vector('полотенце_NOUN'))